In [11]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
from sklearn.feature_selection import SelectKBest, f_classif

In [12]:
# ================= ICT & PRICE ACTION FUNCTIONS =================

def detect_fvg(df):
    """
    Fair Value Gap (FVG): ช่องว่างราคาระหว่างแท่งเทียน
    - Bullish FVG: low[i] > high[i-2] (gap ขึ้น)
    - Bearish FVG: high[i] < low[i-2] (gap ลง)
    """
    high = df['Gold_High'].values
    low = df['Gold_Low'].values
    close = df['Gold_Close'].values
    
    bullish_fvg = np.zeros(len(df))
    bearish_fvg = np.zeros(len(df))
    fvg_size = np.zeros(len(df))
    
    for i in range(2, len(df)):
        # Bullish FVG: low ของแท่งปัจจุบัน > high ของแท่ง 2 แท่งก่อนหน้า
        if low[i] > high[i-2]:
            bullish_fvg[i] = 1
            fvg_size[i] = low[i] - high[i-2]
        
        # Bearish FVG: high ของแท่งปัจจุบัน < low ของแท่ง 2 แท่งก่อนหน้า
        elif high[i] < low[i-2]:
            bearish_fvg[i] = 1
            fvg_size[i] = low[i-2] - high[i]
    
    return pd.DataFrame({
        'Bullish_FVG': bullish_fvg,
        'Bearish_FVG': bearish_fvg,
        'FVG_Size': fvg_size,
        'FVG_Net': bullish_fvg - bearish_fvg  # บวก = Bullish dominance, ลบ = Bearish dominance
    }, index=df.index)


def detect_order_block(df, lookback=10):
    """
    Order Block: แท่งเทียนสุดท้ายก่อนการเคลื่อนไหวแรง
    - Bullish OB: แท่งแดงสุดท้ายก่อนแท่งเขียวที่ break high ก่อนหน้า
    - Bearish OB: แท่งเขียวสุดท้ายก่อนแท่งแดงที่ break low ก่อนหน้า
    """
    high = df['Gold_High'].values
    low = df['Gold_Low'].values
    close = df['Gold_Close'].values
    open_price = df['Gold_Open'].values
    
    bullish_ob = np.zeros(len(df))
    bearish_ob = np.zeros(len(df))
    
    for i in range(lookback, len(df)):
        # หา swing high/low ใน lookback period
        swing_high = np.max(high[i-lookback:i])
        swing_low = np.min(low[i-lookback:i])
        
        # Bullish OB: แท่งปัจจุบันเป็นแท่งเขียวที่ break swing high
        if close[i] > open_price[i] and close[i] > swing_high:
            # หาแท่งแดงสุดท้ายก่อนหน้า
            for j in range(i-1, max(i-lookback, 0), -1):
                if close[j] < open_price[j]:
                    bullish_ob[i] = 1
                    break
        
        # Bearish OB: แท่งปัจจุบันเป็นแท่งแดงที่ break swing low
        elif close[i] < open_price[i] and close[i] < swing_low:
            # หาแท่งเขียวสุดท้ายก่อนหน้า
            for j in range(i-1, max(i-lookback, 0), -1):
                if close[j] > open_price[j]:
                    bearish_ob[i] = 1
                    break
    
    return pd.DataFrame({
        'Bullish_OB': bullish_ob,
        'Bearish_OB': bearish_ob,
        'OB_Net': bullish_ob - bearish_ob
    }, index=df.index)


def detect_bos_choch(df, lookback=20):
    """
    Break of Structure (BOS) & Change of Character (CHoCH)
    - BOS: ราคา break high/low ก่อนหน้า (ต่อเนื่องแนวโน้ม)
    - CHoCH: ราคา break high/low ในทิศทางตรงข้าม (เปลี่ยนแนวโน้ม)
    """
    high = df['Gold_High'].values
    low = df['Gold_Low'].values
    close = df['Gold_Close'].values
    
    bos_bullish = np.zeros(len(df))
    bos_bearish = np.zeros(len(df))
    choch_bullish = np.zeros(len(df))
    choch_bearish = np.zeros(len(df))
    
    # หา swing points อย่างง่าย
    for i in range(lookback, len(df)):
        swing_high = np.max(high[i-lookback:i-1])
        swing_low = np.min(low[i-lookback:i-1])
        
        # แนวโน้มก่อนหน้า (ใช้ slope ของ close)
        prev_trend = np.polyfit(range(lookback), close[i-lookback:i], 1)[0]
        
        # Bullish BOS: break swing high ในแนวโน้มขาขึ้น
        if close[i] > swing_high and prev_trend > 0:
            bos_bullish[i] = 1
        
        # Bearish BOS: break swing low ในแนวโน้มขาลง
        elif close[i] < swing_low and prev_trend < 0:
            bos_bearish[i] = 1
        
        # Bullish CHoCH: break swing high ในแนวโน้มขาลง (เปลี่ยนเป็นขาขึ้น)
        elif close[i] > swing_high and prev_trend < 0:
            choch_bullish[i] = 1
        
        # Bearish CHoCH: break swing low ในแนวโน้มขาขึ้น (เปลี่ยนเป็นขาลง)
        elif close[i] < swing_low and prev_trend > 0:
            choch_bearish[i] = 1
    
    return pd.DataFrame({
        'BOS_Bullish': bos_bullish,
        'BOS_Bearish': bos_bearish,
        'CHoCH_Bullish': choch_bullish,
        'CHoCH_Bearish': choch_bearish,
        'Structure_Score': (bos_bullish + choch_bullish) - (bos_bearish + choch_bearish)
    }, index=df.index)


def detect_candle_patterns(df):
    """
    Candlestick Patterns: รูปแบบแท่งเทียนสำคัญ
    """
    open_price = df['Gold_Open'].values
    high = df['Gold_High'].values
    low = df['Gold_Low'].values
    close = df['Gold_Close'].values
    
    body = np.abs(close - open_price)
    upper_shadow = high - np.maximum(open_price, close)
    lower_shadow = np.minimum(open_price, close) - low
    total_range = high - low
    
    # หลีกเลี่ยงการหารด้วย 0
    total_range = np.where(total_range == 0, 1e-10, total_range)
    
    patterns = {}
    
    # 1. Engulfing
    bullish_engulfing = np.zeros(len(df))
    bearish_engulfing = np.zeros(len(df))
    for i in range(1, len(df)):
        # Bullish Engulfing: แท่งก่อนเป็นแดง, แท่งปัจจุบันเป็นเขียวและใหญ่กว่า
        if (close[i-1] < open_price[i-1] and 
            close[i] > open_price[i] and 
            body[i] > body[i-1] and
            open_price[i] <= close[i-1] and 
            close[i] >= open_price[i-1]):
            bullish_engulfing[i] = 1
        
        # Bearish Engulfing: แท่งก่อนเป็นเขียว, แท่งปัจจุบันเป็นแดงและใหญ่กว่า
        elif (close[i-1] > open_price[i-1] and 
              close[i] < open_price[i] and 
              body[i] > body[i-1] and
              open_price[i] >= close[i-1] and 
              close[i] <= open_price[i-1]):
            bearish_engulfing[i] = 1
    
    patterns['Bullish_Engulfing'] = bullish_engulfing
    patterns['Bearish_Engulfing'] = bearish_engulfing
    
    # 2. Pin Bar / Hammer / Shooting Star
    hammer = np.zeros(len(df))
    shooting_star = np.zeros(len(df))
    for i in range(len(df)):
        # Hammer: lower shadow ยาว, upper shadow สั้น, body เล็ก
        if (lower_shadow[i] > 2 * body[i] and 
            upper_shadow[i] < body[i] and
            body[i] > 0):
            hammer[i] = 1
        
        # Shooting Star: upper shadow ยาว, lower shadow สั้น, body เล็ก
        elif (upper_shadow[i] > 2 * body[i] and 
              lower_shadow[i] < body[i] and
              body[i] > 0):
            shooting_star[i] = 1
    
    patterns['Hammer'] = hammer
    patterns['Shooting_Star'] = shooting_star
    
    # 3. Doji
    doji = np.where(body < 0.1 * total_range, 1, 0)
    patterns['Doji'] = doji
    
    # 4. Inside Bar
    inside_bar = np.zeros(len(df))
    for i in range(1, len(df)):
        if high[i] < high[i-1] and low[i] > low[i-1]:
            inside_bar[i] = 1
    patterns['Inside_Bar'] = inside_bar
    
    # 5. Outside Bar
    outside_bar = np.zeros(len(df))
    for i in range(1, len(df)):
        if high[i] > high[i-1] and low[i] < low[i-1]:
            outside_bar[i] = 1
    patterns['Outside_Bar'] = outside_bar
    
    # 6. Body Size Ratio (relative to total range)
    patterns['Body_Ratio'] = body / total_range
    
    # 7. Upper/Lower Shadow Ratio
    patterns['Upper_Shadow_Ratio'] = upper_shadow / total_range
    patterns['Lower_Shadow_Ratio'] = lower_shadow / total_range
    
    return pd.DataFrame(patterns, index=df.index)


In [ ]:
# ================= MAIN PIPELINE =================

# 1. กำหนด Tickers
tickers = {'Gold': 'GC=F', 'DXY': 'DX-Y.NYB', 'VIX': '^VIX', 'SP500': '^GSPC'}

# 2. ดึงข้อมูล
data_dict = {}
first_index = None

for name, ticker in tickers.items():
    print(f"Downloading {name} ({ticker})...")
    df = yf.download(ticker, period='2y', interval='1h', progress=False)
    
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
        
    if df.empty:
        print(f"⚠️ Warning: No data found for {ticker}")
        continue
        
    if first_index is None:
        first_index = df.index
        
    data_dict[f'{name}_Close'] = df['Close'].squeeze()
    
    if name == 'Gold':
        data_dict['Gold_High'] = df['High'].squeeze()
        data_dict['Gold_Low'] = df['Low'].squeeze()
        data_dict['Gold_Open'] = df['Open'].squeeze()
        data_dict['Gold_Volume'] = df['Volume'].squeeze()

data = pd.DataFrame(data_dict, index=first_index).ffill().dropna()
print(f"✅ Data loaded successfully. Shape: {data.shape}\n")


IndentationError: unexpected indent (3373752249.py, line 1)